# 15 — Agentic GRPO full trainer runbook

Эта тетрадка собирает весь GRPO-контур в одном месте: внешняя подготовка env/model snapshot, profile/topology/preflight, локальный scripted smoke без модели и реальный запуск Tunix `GRPOLearner` через `scripts/run_agentic_grpo.py`.

Дизайн намеренно честный:

- `dry-run` и `scripted-smoke` должны работать локально без загрузки весов;
- real trainer cell запускается только при `RUN_REAL_GRPO=1`;
- веса не скачиваются неявно — snapshot должен быть подготовлен заранее;
- на CPU real Qwen GRPO по умолчанию блокируется, потому что этот путь предназначен для accelerator runner.


## 0. External preparation checklist

Перед real run подготовьте окружение и модель вне notebook:

```bash
pyenv exec python -m uv sync --extra envs --extra prompts --extra tunix --extra dev
# place local Qwen snapshot here, no implicit downloads from the notebook:
# artifacts/models/qwen25-05b-instruct
```

Для локальной проверки без модели достаточно `--dry-run` и `--scripted-smoke`.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

import jax

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

PROFILE = ROOT / 'configs/grpo/qwen_agentic_local.yaml'
CONFIG = ROOT / 'configs/mvp/qwen_craftext.yaml'
TOPOLOGY = ROOT / 'configs/topology/qwen_agentic_grpo_local.yaml'
SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
SCRIPTED_EVIDENCE = ROOT / 'artifacts/runs/agentic-grpo-scripted-smoke-notebook.json'

SPEC = importlib.util.spec_from_file_location('run_agentic_grpo', ROOT / 'scripts/run_agentic_grpo.py')
runner = importlib.util.module_from_spec(SPEC)
assert SPEC and SPEC.loader
SPEC.loader.exec_module(runner)

print('repo:', ROOT)
print('jax backend:', jax.default_backend())
print('profile:', PROFILE)
print('config:', CONFIG, 'exists=', CONFIG.is_file())
print('topology:', TOPOLOGY, 'exists=', TOPOLOGY.is_file())
print('snapshot:', SNAPSHOT, 'exists=', SNAPSHOT.is_dir())


## 1. Validate profile/topology/preflight without model allocation

This uses the same CLI entrypoint as production, but stops before loading actor/reference weights.


In [ ]:
runner.main(['--profile', str(PROFILE), '--dry-run'])


## 2. Run real CrafText agentic tool loop with scripted GRPO generations

This is the local proof that CrafText env construction, MegaPrompts rendering, ToolAgent parsing, Tunix `TrajectoryCollectEngine`, multi-generation grouping and reward normalization are wired before we pay the cost of model allocation.


In [ ]:
runner.main([
    '--profile', str(PROFILE),
    '--scripted-smoke',
    '--scripted-output', str(SCRIPTED_EVIDENCE),
])

payload = json.loads(SCRIPTED_EVIDENCE.read_text(encoding='utf-8'))
print('schema:', payload['schema'])
for generation in payload['generations']:
    print({
        'generation_id': generation['generation_id'],
        'total_reward': generation['total_reward'],
        'advantage': generation['advantage'],
        'done': generation['done'],
        'steps': len(generation['rewards']),
    })


## 3. Real Tunix GRPOLearner run

This cell performs the actual trainer call. It is disabled by default so opening/running the notebook locally cannot accidentally allocate model weights.

Enable it explicitly:

```bash
RUN_REAL_GRPO=1 pyenv exec python -m uv run jupyter lab examples/notebooks/15_agentic_grpo_full_trainer.ipynb
```

Optional debugging-only CPU reproduction:

```bash
RUN_REAL_GRPO=1 ALLOW_CPU_SMOKE=1 ...
```

In production/accelerator mode this path creates `RLCluster(actor, rollout, reference)`, then `GRPOLearner`, and calls `learner.train(...)`.


In [ ]:
RUN_REAL_GRPO = os.environ.get('RUN_REAL_GRPO') == '1'
ALLOW_CPU_SMOKE = os.environ.get('ALLOW_CPU_SMOKE') == '1'

if not RUN_REAL_GRPO:
    print('Skipping real GRPOLearner run. Set RUN_REAL_GRPO=1 to execute this cell.')
elif not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local Qwen snapshot is required for real GRPO: {SNAPSHOT}')
else:
    args = ['--profile', str(PROFILE)]
    if ALLOW_CPU_SMOKE:
        args.append('--allow-cpu-smoke')
    runner.main(args)


## 4. What this proves and what comes next

Passing cells 1–2 proves the critic-free GRPO transport is alive: profile, topology, static preflight, CrafText agentic environment, tool calls, grouped rewards and evidence. Passing cell 3 proves the real Tunix `GRPOLearner` path on prepared hardware/model snapshot.

Next steps for PPO-Lag/CPO should reuse this transport and add explicit value critic and cost critic roles, rather than rebuilding the environment loop.
